<a href="https://colab.research.google.com/github/HasanKhatib/iot-playground/blob/main/spark_ml_lib_without_pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic Survival Prediction using PySpark (No Pipelines)

This guide will help you build a logistic regression model to predict Titanic survival using PySpark.

## Steps:

1. **Setup Environment**: Mount Google Drive and set up Java and Spark.
2. **Initialize Spark Session**: Create a Spark session.
3. **Load Data**: Load training and testing datasets from Google Drive.
4. **Data Preprocessing**: Print schema, show data, and drop rows with missing values.
5. **Feature Engineering**: Convert categorical columns to numerical using `StringIndexer` and assemble features into a single vector.
6. **Prepare Data for Modeling**: Add a dummy 'Survived' column to the test data and rename 'Survived' to 'label' in both datasets.
7. **Train the Model**: Train a logistic regression model.
8. **Make Predictions**: Use the model to make predictions on the test data.
9. **Evaluate the Model**: Evaluate the model's accuracy.

By following these steps, you will be able to build and evaluate a logistic regression model to predict Titanic survival using PySpark.

# Solution

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!pip install -q findspark


In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TitanicSparkLab5").getOrCreate()
print("Spark version:", spark.version)


Spark version: 3.3.0


In [ ]:
train_path = "/content/drive/MyDrive/iot/titanic/train.csv"
test_path  = "/content/drive/MyDrive/iot/titanic/test.csv"

train_df = spark.read.csv(train_path, header=True, inferSchema=True)
test_df  = spark.read.csv(test_path,  header=True, inferSchema=True)

train_df.printSchema()
train_df.show(5)


root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| null|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   

In [ ]:
train_df = train_df.na.drop()
test_df  = test_df.na.drop()


In [ ]:
from pyspark.ml.feature import StringIndexer

sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndexed")
train_df = sex_indexer.fit(train_df).transform(train_df)
test_df  = sex_indexer.fit(test_df).transform(test_df)

embark_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndexed")
train_df = embark_indexer.fit(train_df).transform(train_df)
test_df  = embark_indexer.fit(test_df).transform(test_df)


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["Pclass", "SexIndexed", "Age", "Fare", "EmbarkedIndexed"],
    outputCol="features"
)

train_df = assembler.transform(train_df)
test_df  = assembler.transform(test_df)


In [ ]:
from pyspark.sql.functions import lit

# Add a dummy 'Survived' column to test_df
test_df = test_df.withColumn("Survived", lit(0))


In [ ]:
train_df = train_df.withColumnRenamed("Survived", "label")
test_df  = test_df.withColumnRenamed("Survived", "label")


In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
model = lr.fit(train_df)


In [ ]:
predictions = model.transform(test_df)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
print(predictions)
accuracy = evaluator.evaluate(predictions)
print("Project 1 (No Pipeline) - Test Accuracy =", accuracy)


DataFrame[PassengerId: int, Pclass: int, Name: string, Sex: string, Age: double, SibSp: int, Parch: int, Ticket: string, Fare: double, Cabin: string, Embarked: string, SexIndexed: double, EmbarkedIndexed: double, features: vector, label: int, rawPrediction: vector, probability: vector, prediction: double]
Project 1 (No Pipeline) - Test Accuracy = 0.3218390804597701
